In [11]:
import os
import sys
sys.path.append("../") # go to parent dir
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from PIL import Image
import shutil
import yaml
import torch

# Step 1: Process the original dataset and convert to YOLO format
def process_dataset(df, output_dir='yolo_dataset'):
    """
    Convert the original dataset format to YOLO format.
    
    Args:
        df: pandas DataFrame with columns ['Image_ID', 'class', 'confidence', 'ymin', 'xmin', 'ymax', 'xmax', 'class_id', 'ImagePath']
        output_dir: directory to save the YOLO format dataset
    """
    # Create necessary directories
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'images'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'labels'), exist_ok=True)
    
    # Get unique classes and create class mapping
    classes = df['class'].unique().tolist()
    class_to_id = {class_name: i for i, class_name in enumerate(classes)}
    
    # Save the class mapping to a file
    with open(os.path.join(output_dir, 'classes.txt'), 'w') as f:
        for class_name in classes:
            f.write(f"{class_name}\n")
    
    # Process each image
    processed_images = set()
    for _, row in df.iterrows():
        img_path = row['ImagePath']
        img_id = row['Image_ID']
        
        # Skip if we've already processed this image
        if img_id in processed_images:
            continue
        
        # Copy the image to the YOLO dataset
        img_filename = f"{img_id}.jpg"  # Assuming jpg format, adjust if needed
        dst_img_path = os.path.join(output_dir, 'images', img_filename)
        shutil.copy(img_path, dst_img_path)
        
        # Create a label file for this image
        label_filename = f"{img_id}.txt"
        label_path = os.path.join(output_dir, 'labels', label_filename)
        
        # Get all annotations for this image
        img_annotations = df[df['Image_ID'] == img_id].copy()
        
        # Open the image to get dimensions
        with Image.open(img_path) as img:
            img_width, img_height = img.size
        
        # Write annotations in YOLO format
        with open(label_path, 'w') as f:
            for _, ann in img_annotations.iterrows():
                # Convert bbox coordinates to YOLO format (normalized center x, center y, width, height)
                x_min, y_min = ann['xmin'], ann['ymin']
                x_max, y_max = ann['xmax'], ann['ymax']
                
                # Normalize coordinates
                x_center = ((x_min + x_max) / 2) / img_width
                y_center = ((y_min + y_max) / 2) / img_height
                width = (x_max - x_min) / img_width
                height = (y_max - y_min) / img_height
                
                # Get class ID
                class_id = class_to_id[ann['class']]
                
                # Write in YOLO format: class_id x_center y_center width height
                f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")
        
        processed_images.add(img_id)
    
    print(f"Processed {len(processed_images)} images with {len(df)} annotations.")
    return classes

def split_dataset(output_dir='yolo_dataset', train_ratio=0.7, val_ratio=0.2, test_ratio=0.1):
    """
    Split the dataset into train, validation, and test sets.
    
    Args:
        output_dir: directory containing the YOLO format dataset
        train_ratio: ratio of training data
        val_ratio: ratio of validation data
        test_ratio: ratio of test data
    """
    # Get all image filenames
    image_dir = os.path.join(output_dir, 'images')
    all_images = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
    
    # Create train, val, test directories
    for split in ['train', 'val', 'test']:
        for subdir in ['images', 'labels']:
            os.makedirs(os.path.join(output_dir, split, subdir), exist_ok=True)
    
    # Split the dataset
    train_images, temp_images = train_test_split(all_images, train_size=train_ratio, random_state=42)
    val_size = val_ratio / (val_ratio + test_ratio)
    val_images, test_images = train_test_split(temp_images, train_size=val_size, random_state=42)
    
    # Move files to respective directories
    for split, images in [('train', train_images), ('val', val_images), ('test', test_images)]:
        for img_file in images:
            # Get corresponding label file
            label_file = os.path.splitext(img_file)[0] + '.txt'
            
            # Move image
            src_img = os.path.join(output_dir, 'images', img_file)
            dst_img = os.path.join(output_dir, split, 'images', img_file)
            shutil.copy(src_img, dst_img)
            
            # Move label
            src_label = os.path.join(output_dir, 'labels', label_file)
            dst_label = os.path.join(output_dir, split, 'labels', label_file)
            if os.path.exists(src_label):  # Some images might not have annotations
                shutil.copy(src_label, dst_label)
    
    print(f"Dataset split: {len(train_images)} train, {len(val_images)} validation, {len(test_images)} test images.")

def create_yaml_config(classes, output_dir='yolo_dataset'):
    """
    Create a YAML configuration file for YOLOv5.
    
    Args:
        classes: list of class names
        output_dir: directory containing the YOLO format dataset
    """
    config = {
        'path': os.path.abspath(output_dir),
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images',
        'nc': len(classes),
        'names': classes
    }
    
    with open(os.path.join(output_dir, 'dataset.yaml'), 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    
    print(f"Created YAML configuration file at {os.path.join(output_dir, 'dataset.yaml')}")

# Step 2: Set up and train the YOLO model (YOLOv11)
def train_yolov11(dataset_yaml, model_size='s', epochs=100, batch_size=16, image_size=640):
    """
    Train a YOLOv11 model.
    
    Args:
        dataset_yaml: path to the YAML configuration file
        model_size: YOLOv11 model size ('n', 's', 'm', 'l', 'x')
        epochs: number of training epochs
        batch_size: batch size
        image_size: input image size
    """
    # Install Ultralytics package (which includes YOLOv11)
    os.system('pip install ultralytics')
    
    # Start training with YOLOv11
    import ultralytics
    from ultralytics import YOLO
    
    # Print Ultralytics version for reference
    print(f"Ultralytics version: {ultralytics.__version__}")
    
    # Load the base model
    model = YOLO(f'yolov11{model_size}.pt')
    
    # Train the model using the dataset YAML config
    model.train(
        data=dataset_yaml,
        epochs=epochs,
        batch=batch_size,
        imgsz=image_size,
        patience=50,  # Early stopping patience
        device='0',   # Use GPU 0 or 'cpu' if no GPU
        cache=True,
        project='yolov11_plant_detection',
        name=f'yolov11{model_size}_run1',
        save=True,    # Save best model
        pretrained=True,
        verbose=True
    )

# Step 3: Run inference on test images without bounding boxes
def run_inference(model_path, test_image_dir, output_dir='predictions', conf_threshold=0.25):
    """
    Run inference on test images using YOLOv11.
    
    Args:
        model_path: path to the trained YOLOv11 model
        test_image_dir: directory containing test images
        output_dir: directory to save prediction results
        conf_threshold: confidence threshold for detections
    """
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Load model using Ultralytics YOLO
    from ultralytics import YOLO
    model = YOLO(model_path)
    
    # Set confidence threshold
    # Note: In YOLOv11, this is done at prediction time
    
    # Run inference
    results = model.predict(
        source=test_image_dir,
        conf=conf_threshold,
        save=True,
        save_txt=True,
        save_conf=True,
        project=output_dir,
        name='detect',
        verbose=True
    )
    
    # Export results to a CSV file
    predictions = []
    
    for result in results:
        img_name = os.path.basename(result.path)
        boxes = result.boxes
        
        if len(boxes) > 0:
            # Extract coordinates, confidence, and class
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()  # xyxy format (top-left, bottom-right)
                conf = box.conf.item()
                cls = int(box.cls.item())
                class_name = model.names[cls]
                
                predictions.append({
                    'image_name': img_name,
                    'class': class_name,
                    'confidence': conf,
                    'xmin': x1,
                    'ymin': y1,
                    'xmax': x2,
                    'ymax': y2
                })
    
    # Create DataFrame and save
    if predictions:
        pred_df = pd.DataFrame(predictions)
        pred_df.to_csv(os.path.join(output_dir, 'predictions.csv'), index=False)
    
    print(f"Inference complete. Results saved to {output_dir}")

# Step 4: Aggregate image-level predictions for multi-label evaluation
def aggregate_predictions(predictions_csv, output_csv='image_level_predictions.csv'):
    """
    Aggregate bounding box predictions to image-level class predictions.
    
    Args:
        predictions_csv: path to CSV file with bounding box predictions
        output_csv: path to save image-level predictions
    """
    # Load predictions
    pred_df = pd.read_csv(predictions_csv)
    
    # Group by image and aggregate classes
    image_level = pred_df.groupby('image_name')['class'].apply(lambda x: list(set(x))).reset_index()
    image_level.rename(columns={'class': 'predicted_classes'}, inplace=True)
    
    # Save to CSV
    image_level.to_csv(output_csv, index=False)
    
    print(f"Aggregated predictions saved to {output_csv}")


In [13]:
from functions.loading_functions import *

In [15]:
train_df = get_dataset("train")

In [16]:
# Process dataset
classes = process_dataset(train_df)

Processed 5529 images with 9792 annotations.


In [ ]:
split_dataset()
create_yaml_config(classes)

# Train model using YOLOv11
train_yolov11('yolo_dataset/dataset.yaml', model_size='s', epochs=100)

['healthy', 'anthracnose', 'cssvd']

In [ ]:


# Run inference on test images
test_image_dir = 'test_images_without_bbox'  # Your test images directory
model_path = 'yolov11_plant_detection/yolov11s_run1/weights/best.pt'
run_inference(model_path, test_image_dir)

# Aggregate predictions for multi-label evaluation
aggregate_predictions('predictions/detect/predictions.csv')

if __name__ == '__main__':
main()

In [ ]:

# Example usage
if __name__ == "__main__":
    # Prepare test data
    gt_csv = prepare_test_data("test_data.csv", "test_images")
    
    # Visualize predictions
    batch_visualize_predictions("test_images", "predictions/predictions.csv", "visualizations")
    
    # Prepare submission
    submission_csv = prepare_submission("predictions/predictions.csv", "test_images", "submission.csv")
    
    # Evaluate predictions
    from evaluation_script import evaluate_multilabel
    evaluate_multilabel(gt_csv, submission_csv)